In [22]:
from datasets import load_dataset
import os, json, time, random, pathlib
import requests
import numpy as np
import pandas as pd
import torch
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)


In [2]:
os.environ["HF_TOKEN"] = input()

In [ ]:
dataset = load_dataset("pameydorke/arcanum-cross-platform-queries-synthetic")

README.md:   0%|          | 0.00/384 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 17.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1043 [00:00<?, ? examples/s]

In [6]:
DATA_URL = "https://raw.githubusercontent.com/ankorn/arcanumsearch/refs/heads/main/data/retriever_test_ds.json"
test_ds = requests.get(DATA_URL, timeout=30).json()

In [7]:
FT_MODEL_ID   = "pameydorke/arcanum-cross-platform-retriever"
BASE_MODEL_ID = "BAAI/bge-m3"

In [ ]:
base_model = SentenceTransformer('BAAI/bge-m3')
ft_model = SentenceTransformer('pameydorke/arcanum-cross-platform-retriever')

base_model.eval()
ft_model.eval()

modules.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/284 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/62.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

In [ ]:
from collections import defaultdict

queries = {}
corpus = {}
relevant_docs = {}
url2text = {}
unique_docs = {}

# Build corpus from train
for example in dataset["train"]:
    url = example["url"]
    doc_text = example["content"]
    url2text[url] = doc_text

    if doc_text not in unique_docs:
        doc_id = f"doc_{len(unique_docs)}"
        unique_docs[doc_text] = doc_id
        corpus[doc_id] = doc_text

query_text = {}
query_to_docs = defaultdict(set)
skipped_counter = 0

for example in test_ds:
    query = example["query"]
    url = example["url"]

    if url not in url2text:
        skipped_counter += 1
        continue

    doc_text = url2text[url]

    if doc_text not in unique_docs:
        skipped_counter += 1
        continue

    doc_id = unique_docs[doc_text]

    # Group by query, not by row
    qid = example.get("query_id", query)

    if qid not in query_text:
        query_text[qid] = query

    query_to_docs[qid].add(doc_id)

queries = {qid: query_text[qid] for qid in query_to_docs}
relevant_docs = {qid: docs for qid, docs in query_to_docs.items() if docs}

print("Skipped:", skipped_counter)
print("Queries:", len(queries))
print("Avg relevant docs/query:", sum(len(v) for v in relevant_docs.values()) / max(len(relevant_docs), 1))

In [ ]:
evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="arcanum-retriever-test"
)

In [24]:
base_results = evaluator(base_model)

In [25]:
base_results

{'arcanum-retriever-test_cosine_accuracy@1': 0.6338028169014085,
 'arcanum-retriever-test_cosine_accuracy@3': 0.7676056338028169,
 'arcanum-retriever-test_cosine_accuracy@5': 0.823943661971831,
 'arcanum-retriever-test_cosine_accuracy@10': 0.8661971830985915,
 'arcanum-retriever-test_cosine_precision@1': 0.6338028169014085,
 'arcanum-retriever-test_cosine_precision@3': 0.25586854460093894,
 'arcanum-retriever-test_cosine_precision@5': 0.16478873239436617,
 'arcanum-retriever-test_cosine_precision@10': 0.08661971830985914,
 'arcanum-retriever-test_cosine_recall@1': 0.6338028169014085,
 'arcanum-retriever-test_cosine_recall@3': 0.7676056338028169,
 'arcanum-retriever-test_cosine_recall@5': 0.823943661971831,
 'arcanum-retriever-test_cosine_recall@10': 0.8661971830985915,
 'arcanum-retriever-test_cosine_ndcg@10': 0.7487667975216589,
 'arcanum-retriever-test_cosine_mrr@10': 0.7111083165660631,
 'arcanum-retriever-test_cosine_map@100': 0.7148435587487354}

In [28]:
ft_results = evaluator(ft_model)

ft_results

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [01:31<00:00, 91.09s/it]


{'arcanum-retriever-test_cosine_accuracy@1': 0.6408450704225352,
 'arcanum-retriever-test_cosine_accuracy@3': 0.7676056338028169,
 'arcanum-retriever-test_cosine_accuracy@5': 0.8098591549295775,
 'arcanum-retriever-test_cosine_accuracy@10': 0.8802816901408451,
 'arcanum-retriever-test_cosine_precision@1': 0.6408450704225352,
 'arcanum-retriever-test_cosine_precision@3': 0.25586854460093894,
 'arcanum-retriever-test_cosine_precision@5': 0.16197183098591544,
 'arcanum-retriever-test_cosine_precision@10': 0.08802816901408449,
 'arcanum-retriever-test_cosine_recall@1': 0.6408450704225352,
 'arcanum-retriever-test_cosine_recall@3': 0.7676056338028169,
 'arcanum-retriever-test_cosine_recall@5': 0.8098591549295775,
 'arcanum-retriever-test_cosine_recall@10': 0.8802816901408451,
 'arcanum-retriever-test_cosine_ndcg@10': 0.7571203441101373,
 'arcanum-retriever-test_cosine_mrr@10': 0.7183266264252179,
 'arcanum-retriever-test_cosine_map@100': 0.7222959294567696}